# HRF Feature Analysis

### Imports

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import f_oneway

sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd().parent / "benchmark"))
from maskel.config import ExtractionConfig, OutputConfig, PipelineConfig
from maskel.pipeline import analyze_segmentation_mask

from hrf import HRFDataset, preprocess_segmentation

### Load Data

In [ ]:
# Init HRF dataset
dataset = HRFDataset("../data/HRF")

# Config with ALL features and cleanup enabled
config = PipelineConfig(
    extraction=ExtractionConfig(
        branches=True,
        nodes=True,
        summary=True,
        fractal_dimension=True,
        mask_radius=True,
        junction_cleanup=True,
        cleanup_threshold_factor=2.5,
        closing_iterations=0,
        fill_holes=True,
        max_hole_size=300,
    ),
    output=OutputConfig(),
)

# Process all 45 HRF samples through maskel with full feature extraction
results = []
all_branch_records = {}
all_node_records = {}
for i in range(len(dataset)):
    print(f"Processing sample {i + 1}/{len(dataset)}")
    _, seg, mask, info = dataset.load_sample(i)
    binary = preprocess_segmentation(seg, mask, min_size=50)
    result = analyze_segmentation_mask(binary, config)
    row = {"image": info["name"], **result.summary_features[0]}
    results.append(row)
    if result.branch_records:
        all_branch_records[info["name"]] = pd.DataFrame(result.branch_records)
    if result.node_records:
        all_node_records[info["name"]] = pd.DataFrame(result.node_records)

df = pd.DataFrame(results).set_index("image")

# Stack branch/node records across images for multi-image analysis
branch_df = pd.concat(
    [d.assign(image=name) for name, d in all_branch_records.items()],
    ignore_index=True,
)
node_df = pd.concat(
    [d.assign(image=name) for name, d in all_node_records.items()],
    ignore_index=True,
)

phenotype_map = {"h": "healthy", "dr": "diabetes", "g": "glaucoma"}
sample_meta = pd.DataFrame(index=df.index)
sample_meta["sample_suffix"] = (
    sample_meta.index.to_series().astype(str).str.split("_").str[-1]
)
sample_meta["phenotype"] = (
    sample_meta["sample_suffix"].map(phenotype_map).fillna("unknown")
)

# Map phenotype labels onto branch and node records
phenotype_by_image = sample_meta["phenotype"].to_dict()
branch_df["phenotype"] = branch_df["image"].map(phenotype_by_image)
node_df["phenotype"] = node_df["image"].map(phenotype_by_image)

print(f"\nSummary features: {df.shape[0]} samples x {df.shape[1]} features")
print(
    f"Branch records:   {len(branch_df)} rows across {branch_df['image'].nunique()} images"
)
print(
    f"Node records:     {len(node_df)} rows across {node_df['image'].nunique()} images"
)
print(sample_meta["phenotype"].value_counts().sort_index().to_string())

### Inspect it

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

### Phenotype summary

In [ ]:
summary = (
    df.join(sample_meta["phenotype"]).groupby("phenotype").agg(["mean", "std", "count"])
)
summary.columns = ["_".join(col) for col in summary.columns]
summary.reset_index(inplace=True)
summary

### Feature distributions

In [ ]:
analysis_df = df.join(sample_meta["phenotype"])
variance_order = df.var().sort_values(ascending=False)
plot_features = variance_order.head(min(8, len(variance_order))).index.tolist()

fig, axes = plt.subplots(len(plot_features), 1, figsize=(12, 3.2 * len(plot_features)))
if len(plot_features) == 1:
    axes = [axes]

for ax, feature in zip(axes, plot_features):
    sns.boxplot(data=analysis_df, x="phenotype", y=feature, ax=ax, palette="Set2")
    ax.set_title(feature)
    ax.set_xlabel("")
    ax.set_ylabel("Value")
    ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.savefig("./figures/feature_distributions.png", dpi=300, bbox_inches="tight")
plt.show()

### Correlation

In [ ]:
corr_spearman = df.corr(method="spearman")

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_spearman, dtype=bool), k=1)
sns.heatmap(
    corr_spearman,
    annot=False,
    mask=mask,
    cmap="coolwarm",
    center=0,
    ax=ax,
    cbar_kws={"label": "Spearman rho"},
)
ax.set_title("Spearman correlation")

plt.tight_layout()
plt.savefig("./figures/correlation_heatmap.svg", bbox_inches="tight")
plt.show()

In [ ]:
print("strongly correlated (abs(rho) > 0.8)")
upper = corr_spearman.where(np.triu(np.ones(corr_spearman.shape, dtype=bool), k=1))
upper.stack().rename("Spearman rho").rename_axis(["Feature 1", "Feature 2"]).loc[
    lambda s: s.abs() > 0.8
].sort_values(key=np.abs, ascending=False).reset_index()

### ANOVA Feature Difference

In [ ]:
grouped = [group for _, group in analysis_df.groupby("phenotype")]

anova_rows = []
for feature in df.columns:
    feature_values = [group[feature].dropna() for group in grouped]
    f_stat, p_value = f_oneway(*feature_values)
    anova_rows.append({"Feature": feature, "F": f_stat, "p_value": p_value})

anova_df = pd.DataFrame(anova_rows).sort_values(
    ["p_value", "F"], ascending=[True, False]
)

top_anova = anova_df.head(10).sort_values("F")
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_anova["Feature"], top_anova["F"], color="#4c78a8", edgecolor="black")
ax.set_title("Top feature differences by phenotype (ANOVA F-statistic)")
ax.set_xlabel("F-statistic")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.savefig("./figures/feature_difference_anova.svg", bbox_inches="tight")
plt.show()

In [ ]:
anova_df.head(10)

In [ ]:
top_features = anova_df.head(10)["Feature"].tolist()
phenotype_order = ["healthy", "glaucoma", "diabetes"]
mean_by_phenotype = analysis_df.groupby("phenotype")[top_features].agg(["mean", "std"])
present_phenotypes = [
    phenotype for phenotype in phenotype_order if phenotype in mean_by_phenotype.index
]
formatted_mean_by_phenotype = pd.DataFrame(index=top_features)
for phenotype in present_phenotypes:
    formatted_mean_by_phenotype[phenotype] = [
        f"{mean_by_phenotype.loc[phenotype, (feature, 'mean')]:.3f} ± {mean_by_phenotype.loc[phenotype, (feature, 'std')]:.3f}"
        for feature in top_features
    ]
formatted_mean_by_phenotype.index.name = "feature"
formatted_mean_by_phenotype